# Vietnamese Noise-Aware Hate Speech Detection

Notebook minh hoạ toàn bộ pipeline: tiền xử lý dữ liệu tiếng Việt có nhiễu,
kiến trúc mô hình đa nhiệm (multi-task) dựa trên XLM-RoBERTa, và ensemble
soft-voting 3 seed để dự đoán 2 nhãn cùng lúc:

1. **Hate label** — `CLEAN` / `OFFENSIVE` / `HATE`
2. **Noise type** — loại nhiễu phát hiện được trong câu (teencode, mất dấu, lặp ký tự...)

> **Lưu ý:** 3 mô hình đã được huấn luyện sẵn (xem mục 6). Notebook này giữ
> nguyên đầy đủ code huấn luyện để người đọc hiểu rõ nguyên lý, **không
> cần chạy lại** — phần demo ở cuối dùng thẳng checkpoint có sẵn để suy luận
> trên vài mẫu dữ liệu thật.


## 0. Cài đặt thư viện & Import

Các thư viện cần thiết: `transformers` (XLM-RoBERTa), `pyvi` (tách từ tiếng
Việt), `torch`, `scikit-learn` (tính F1).


## 0.Cho BASELINE ghi nhận tên 3 model SEEDS



In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install transformers scikit-learn pandas numpy torch pyvi

import os
import re
import random
import unicodedata
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaModel, XLMRobertaTokenizer, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score
from pyvi import ViTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Đang sử dụng thiết bị tính toán: {device}")

## 1 .Import model từ googleDrive vào
**Lưu ý:** Nội dung phải được chỉnh chỉnh theo Path Drive chính mà bạn set up, chú ý file public_test.csv nên upload theo đường dẫn /content/public_test.csv

In [ ]:
from google.colab import drive
import os
import shutil

# 1. Kết nối với Google Drive
drive.mount('/content/drive')

# 2. Tạo thư mục chứa model trên Drive
# Phần sửa đổi thư mục drive để chạy demo.
DRIVE_DIR = '/content/drive/MyDrive/FPTU_OLP_ALL_TASK/NLP_Data'#Nên chỉnh sửa lại phù hợp theo link drive của bạn
os.makedirs(DRIVE_DIR, exist_ok=True)

# 3. Copy các model đã train xong lên Drive (nếu chưa có trên Drive)
for seed in SEEDS:
    local_path = f'/content/best_model_seed{seed}.pth'
    drive_path = f'{DRIVE_DIR}/best_model_seed{seed}.pth'

    if os.path.exists(local_path) and not os.path.exists(drive_path):
        print(f"[*] Đang copy {local_path} -> Drive...")
        shutil.copy(local_path, drive_path)
        print("[+] Copy thành công!")

# 4. Cập nhật lại đường dẫn model để trỏ tới Drive
MODEL_PATHS = [
    f"{DRIVE_DIR}/best_model_seed2026.pth",
    f"{DRIVE_DIR}/best_model_seed2027.pth",
    f"{DRIVE_DIR}/best_model_seed2028.pth",
]
print("\n[*] Đường dẫn MODEL_PATHS hiện tại đã được phân luồng sang Drive:")
for p in MODEL_PATHS:
    print("  -", p)

## 2. Nhãn chuẩn & Seed

Ánh xạ nhãn văn bản (`"HATE"`, `"TEENCODE"`...) sang chỉ số nguyên để đưa vào
mô hình, và hàm cố định seed để đảm bảo khả năng tái lập kết quả.


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

LABEL_TO_ID_HATE = {"CLEAN": 0, "OFFENSIVE": 1, "HATE": 2}
LABEL_TO_ID_NOISE = {
    "ORIGINAL": 0, "NO_DIACRITICS": 1, "TEENCODE": 2,
    "CHAR_REPEAT": 3, "PUNCT_NOISE": 4, "OBFUSCATION": 5, "MIXED": 6
}
STRATEGY_TO_ID = {
    "original": 0, "no_diacritics": 1, "teencode": 2,
    "char_repeat": 3, "punct_noise": 4, "obfuscate": 5, "mixed": 6
}
HATE_MAP = {v: k for k, v in LABEL_TO_ID_HATE.items()}
NOISE_MAP = {v: k for k, v in LABEL_TO_ID_NOISE.items()}

## 3. Tiền xử lý & Tạo nhiễu (`TextAugmenter`)

**Nguyên lý cốt lõi của cả pipeline nằm ở đây.** Class này có 2 vai trò tách
biệt rõ ràng:

- `preprocess()` — bước làm sạch **DUY NHẤT**, dùng chung tuyệt đối cho cả
  train / validation / test: xoá URL/HTML, chuẩn hoá Unicode (NFC), rồi tách
  từ tiếng Việt bằng `ViTokenizer` (ví dụ `"rất vui"` → `"rất_vui"`).
- `apply_noise()` — chỉ áp dụng cho **dữ liệu train**, mô phỏng 5 kiểu nhiễu
  thường gặp trên mạng xã hội (teencode, mất dấu, lặp ký tự, che ký tự, hoặc
  trộn nhiều kiểu) để mô hình học được cách "miễn nhiễm" trước nhiễu thật.

> ⚠️ **Bài học rút ra khi debug dự án này:** phiên bản đầu tiên từng dùng 2
> hàm làm sạch khác nhau cho train và test (dữ liệu train qua `preprocess()`,
> test qua một hàm `denoise()` khác hẳn) → khiến mô hình học một phân bố
> input nhưng bị đánh giá trên phân bố khác, kéo điểm F1 xuống rất thấp.
> Đây là lý do vì sao **chỉ có một hàm `preprocess()` duy nhất** được dùng
> xuyên suốt notebook này.


In [ ]:
class TextAugmenter:
    def __init__(self, aug_prob=0.15):
        self.aug_prob = aug_prob
        self.teencode_dict = {
            "không": "ko", "được": "dc", "người": "ng", "anh": "a", "em": "e",
            "chào": "hi", "gì": "j", "thích": "thik", "quá": "wá", "rồi": "r",
            "biết": "bit", "vậy": "z", "với": "vs", "nhưng": "nhg", "nhiều": "nhìu",
            "chồng": "ck", "vợ": "vk", "luôn": "lun", "rất": "rớt", "thế": "thía"
        }
        self.sensitive_keywords = {"chửi", "ngu", "đần", "chó", "cút", "địt", "lồn", "cặc"}
        self.vietnamese_accents = {
            'a': 'áàảãạâấầẩẫậăắằẳẵặ', 'd': 'đ', 'e': 'éèẻẽẹêếềểễệ',
            'i': 'íìỉĩị', 'o': 'óòỏõọôốồổỗộơớờởỡợ', 'u': 'úùủũụưứừửữự', 'y': 'ýỳỷỹỵ'
        }
        self.remove_accent_map = {}
        for non_accent, accents in self.vietnamese_accents.items():
            for accent in accents:
                self.remove_accent_map[accent] = non_accent
                self.remove_accent_map[accent.upper()] = non_accent.upper()

    def preprocess(self, text):
        """Bước làm sạch DUY NHẤT - dùng chung cho train / val / test."""
        text = re.sub(r'http\S+|www\S+|https\S+', '', str(text), flags=re.MULTILINE)
        text = re.sub(r'<.*?>', '', text)
        text = unicodedata.normalize('NFC', text).strip()
        return ViTokenizer.tokenize(text)

    def is_sensitive(self, word):
        parts = word.lower().split('_')
        return any(p in self.sensitive_keywords for p in parts)

    def apply_teencode(self, text):
        words = text.split()
        out = []
        for w in words:
            clean_w = w.lower().replace('_', ' ')
            if not self.is_sensitive(w) and clean_w in self.teencode_dict and random.random() < self.aug_prob:
                out.append(self.teencode_dict[clean_w].replace(' ', '_'))
            else:
                out.append(w)
        return " ".join(out)

    def remove_diacritics(self, text):
        return " ".join(["".join([self.remove_accent_map.get(c, c) for c in w])
                          if not self.is_sensitive(w) and random.random() < self.aug_prob else w
                          for w in text.split()])

    def char_repeat(self, text):
        return " ".join([w + (w[-1] * random.randint(1, 3))
                          if len(w) > 0 and w[-1].isalpha() and not self.is_sensitive(w) and random.random() < self.aug_prob else w
                          for w in text.split()])

    def obfuscate(self, text):
        out = []
        for w in text.split():
            if not self.is_sensitive(w):
                chars = list(w)
                for i in range(len(chars)):
                    if chars[i].isalpha() and random.random() < (self.aug_prob * 0.3):
                        chars[i] = '*'
                w = "".join(chars)
            out.append(w)
        return " ".join(out)

    def apply_noise(self, text, strategy="mixed"):
        """Áp dụng 1 kiểu nhiễu lên text ĐÃ được preprocess() từ trước."""
        if strategy == "teencode":
            return self.apply_teencode(text)
        elif strategy == "no_diacritics":
            return self.remove_diacritics(text)
        elif strategy == "char_repeat":
            return self.char_repeat(text)
        elif strategy == "obfuscate":
            return self.obfuscate(text)
        elif strategy == "mixed":
            if random.random() < 0.5: text = self.apply_teencode(text)
            if random.random() < 0.4: text = self.char_repeat(text)
            if random.random() < 0.3: text = self.remove_diacritics(text)
        return text

    def augment(self, text, strategy="mixed"):
        """Tương thích ngược: preprocess + apply_noise trong 1 lần gọi."""
        text = self.preprocess(text)
        return self.apply_noise(text, strategy)

### Thử nhanh `TextAugmenter` trên 1 câu ví dụ

Chạy thử để thấy trực quan input trước/sau khi tiền xử lý và thêm nhiễu —
không cần dữ liệu train, chỉ minh hoạ nguyên lý.


In [ ]:
demo_augmenter = TextAugmenter(aug_prob=0.5)
sample_text = "Hôm nay trời rất đẹp, mọi người ơi đi chơi không?"

clean = demo_augmenter.preprocess(sample_text)
print("Gốc:          ", sample_text)
print("Sau preprocess:", clean)
for strat in ["teencode", "char_repeat", "no_diacritics", "mixed"]:
    print(f"Sau '{strat}':", demo_augmenter.apply_noise(clean, strategy=strat))

## 4. Dataset Wrappers

Hai class Dataset dùng chung `preprocess()` ở trên:

- `RViHSDDataset` — dùng cho train/val. Tham số `is_train=True` bật augment
  nhiễu ngẫu nhiên (50% xác suất mỗi mẫu); `is_train=False` (validation) chỉ
  làm sạch, không thêm nhiễu — mô phỏng đúng dữ liệu thật khi đánh giá.
- `TestDataset` — dùng khi suy luận (không có nhãn), chỉ cần `preprocess()`.


In [ ]:
class RViHSDDataset(Dataset):
    def __init__(self, texts, hate_labels, noise_labels, tokenizer, max_len=128,
                 augmenter=None, is_train=False):
        self.texts = texts
        self.hate_labels = hate_labels
        self.noise_labels = noise_labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.augmenter = augmenter
        self.is_train = is_train

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        hate_lbl = self.hate_labels[item]
        noise_lbl = self.noise_labels[item]

        if self.augmenter is not None:
            text = self.augmenter.preprocess(text)
            if self.is_train and random.random() < 0.5:
                strategy = random.choice(["teencode", "no_diacritics", "char_repeat", "obfuscate", "mixed"])
                text = self.augmenter.apply_noise(text, strategy=strategy)
                noise_lbl = STRATEGY_TO_ID[strategy]

        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'hate_label': torch.tensor(hate_lbl, dtype=torch.long),
            'noise_label': torch.tensor(noise_lbl, dtype=torch.long)
        }


class TestDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128, cleaner=None):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.cleaner = cleaner

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = self.cleaner.preprocess(str(self.texts[item])) if self.cleaner else str(self.texts[item])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }

## 5. Kiến trúc mô hình

```
XLM-RoBERTa (backbone, đóng vai trò trích xuất đặc trưng ngữ nghĩa)
        │
        ├──► Noise Head (multi-sample dropout) ──► noise_logits (7 lớp)
        │                                              │
        │                                     softmax + projection (7→32)
        │                                              │
        └──► concat(CLS_embedding, noise_features) ──► Hate Head ──► hate_logits (3 lớp)
```

**Ý tưởng chính:** nhánh `noise` được tính trước, rồi đặc trưng của nó
(`noise_features`) được nối (concat) vào biểu diễn CLS trước khi đưa vào
nhánh `hate`. Nói cách khác, **mô hình "biết" câu này có khả năng bị nhiễu
loại gì trước khi đưa ra phán đoán về mức độ thù ghét** — đây là lý do kiến
trúc được đặt tên "Noise-Aware".

**Multi-Sample Dropout:** thay vì chỉ dùng 1 tầng dropout, mô hình dùng 5
tầng dropout với tỉ lệ khác nhau (0.1 → 0.5), tính logits qua cả 5 rồi lấy
trung bình — một kỹ thuật giúp giảm phương sai và ổn định hơn so với dropout
đơn lẻ, gần giống hiệu ứng ensemble ngay bên trong 1 model.


In [ ]:
class ComplexNoiseAwareFPTUModel(nn.Module):
    def __init__(self, model_name="xlm-roberta-base"):
        super().__init__()
        self.roberta = XLMRobertaModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size

        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in np.linspace(0.1, 0.5, 5)])

        self.noise_dense = nn.Linear(hidden_size, hidden_size)
        self.noise_out = nn.Linear(hidden_size, 7)

        self.noise_projection = nn.Linear(7, 32)
        self.layer_norm_noise = nn.LayerNorm(32)

        self.hate_dense = nn.Linear(hidden_size + 32, hidden_size)
        self.hate_out = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_repr = outputs.last_hidden_state[:, 0, :]

        noise_logits = 0
        for dropout in self.dropouts:
            x = dropout(cls_repr)
            x = F.gelu(self.noise_dense(x))
            noise_logits += self.noise_out(x)
        noise_logits = noise_logits / len(self.dropouts)

        noise_probs = torch.softmax(noise_logits, dim=-1)
        noise_features = self.noise_projection(noise_probs)
        noise_features = self.layer_norm_noise(noise_features)

        combined_repr = torch.cat((cls_repr, noise_features), dim=-1)

        hate_logits = 0
        for dropout in self.dropouts:
            x = dropout(combined_repr)
            x = F.gelu(self.hate_dense(x))
            hate_logits += self.hate_out(x)
        hate_logits = hate_logits / len(self.dropouts)

        return hate_logits, noise_logits

## 6. Hàm mất mát (Loss)

- **Focal Loss** cho nhánh `hate`, có trọng số `alpha=[0.2, 0.3, 0.5]` ưu
  tiên lớp thiểu số (`HATE` hiếm gặp hơn `CLEAN` rất nhiều trong dữ liệu
  thật) — giúp mô hình không "lười" chỉ đoán toàn `CLEAN`.
- **Cross-Entropy + label smoothing** cho nhánh `noise`.
- **Uncertainty Weighting** (`log_vars` là tham số học được) — thay vì cộng
  2 loss với trọng số cố định, mô hình **tự học** nên tin tưởng nhánh nào
  hơn qua từng giai đoạn huấn luyện.


In [ ]:
class DynamicFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        return focal_loss.mean()


class AdvancedLossWrapper(nn.Module):
    def __init__(self, num_tasks=2, device='cpu'):
        super().__init__()
        self.log_vars = nn.Parameter(torch.full((num_tasks,), -0.5))
        alpha_weights = torch.tensor([0.2, 0.3, 0.5]).to(device)
        self.focal_loss_hate = DynamicFocalLoss(alpha=alpha_weights, gamma=2.0)
        self.cross_entropy_noise = nn.CrossEntropyLoss(label_smoothing=0.1)

    def forward(self, hate_logits, noise_logits, hate_targets, noise_targets):
        loss_hate = self.focal_loss_hate(hate_logits, hate_targets)
        loss_noise = self.cross_entropy_noise(noise_logits, noise_targets)
        loss_0 = torch.exp(-self.log_vars[0]) * loss_hate + self.log_vars[0]
        loss_1 = torch.exp(-self.log_vars[1]) * loss_noise + self.log_vars[1]
        return loss_0 + loss_1

## 7. Đánh giá & Vòng lặp huấn luyện Ensemble

Điểm số được tính theo đúng công thức chấm điểm của đề bài:

```
Score = 0.85 × F1_hate(macro) + 0.15 × F1_noise(macro)
```

Ensemble gồm **3 model độc lập** (seed `2026`, `2027`, `2028`), mỗi model
huấn luyện 10 epoch, chỉ checkpoint tốt nhất (Score cao nhất trên tập
validation) của mỗi seed được giữ lại.

> ### ⚠️ KHÔNG CẦN CHẠY LẠI CELL DƯỚI ĐÂY
> 3 checkpoint đã được huấn luyện sẵn và đính kèm trong thư mục `models/`
> (xem hướng dẫn tải ở `README.md`). Cell này được giữ lại **chỉ để minh
> hoạ logic huấn luyện** — nếu chạy, nó đòi hỏi `training_set.csv` /
> `validation_set.csv` (không public do quy định bản quyền đề thi) và mất
> khoảng 30-40 phút/seed trên GPU. Muốn xem kết quả ngay, chuyển thẳng
> xuống **Mục 7 — Demo suy luận**.


In [ ]:
def evaluate(model, val_dataloader, device):
    model.eval()
    hate_preds, hate_targets, noise_preds, noise_targets = [], [], [], []
    with torch.no_grad():
        for batch in val_dataloader:
            input_ids, attn_mask = batch['input_ids'].to(device), batch['attention_mask'].to(device)
            h_tgt, n_tgt = batch['hate_label'].to(device), batch['noise_label'].to(device)

            h_logits, n_logits = model(input_ids, attn_mask)
            hate_preds.extend(torch.argmax(h_logits, dim=1).cpu().numpy())
            hate_targets.extend(h_tgt.cpu().numpy())
            noise_preds.extend(torch.argmax(n_logits, dim=1).cpu().numpy())
            noise_targets.extend(n_tgt.cpu().numpy())

    h_f1 = f1_score(hate_targets, hate_preds, average='macro')
    n_f1 = f1_score(noise_targets, noise_preds, average='macro')
    return h_f1, n_f1, 0.85 * h_f1 + 0.15 * n_f1


def train_one_model(seed, train_dataloader, val_dataloader, epochs=10, accumulation_steps=2,
                     save_dir="models"):
    print(f"\n=== Huấn luyện model với SEED={seed} ===")
    seed_everything(seed)

    model = ComplexNoiseAwareFPTUModel(model_name="xlm-roberta-base").to(device)
    loss_fn = AdvancedLossWrapper(num_tasks=2, device=device).to(device)

    total_steps = (len(train_dataloader) // accumulation_steps) * epochs
    optimizer = AdamW(list(model.parameters()) + list(loss_fn.parameters()), lr=4e-5, weight_decay=0.01)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    scaler = torch.amp.GradScaler(device.type)

    best_f1 = 0.0
    os.makedirs(save_dir, exist_ok=True)
    model_save_path = os.path.join(save_dir, f'best_model_seed{seed}.pth')

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_dataloader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            hate_targets = batch['hate_label'].to(device)
            noise_targets = batch['noise_label'].to(device)

            with torch.autocast(device_type=device.type):
                hate_logits, noise_logits = model(input_ids, attention_mask)
                loss = loss_fn(hate_logits, noise_logits, hate_targets, noise_targets)
                loss = loss / accumulation_steps

            scaler.scale(loss).backward()

            if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_dataloader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            total_train_loss += loss.item() * accumulation_steps

        h_f1, n_f1, combined_f1 = evaluate(model, val_dataloader, device)
        avg_loss = total_train_loss / len(train_dataloader)
        print(f"[seed {seed}] Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Hate F1: {h_f1:.4f} | Noise F1: {n_f1:.4f} | Score: {combined_f1:.4f}")

        if combined_f1 > best_f1:
            print(f" -> Điểm tăng! Đang lưu mô hình (seed {seed})...")
            best_f1 = combined_f1
            torch.save(model.state_dict(), model_save_path)

    return model_save_path, best_f1

# ĐỂ CHẠY THẬT (cần training_set.csv / validation_set.csv):
#
# tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")
# augmenter = TextAugmenter(aug_prob=0.15)
# train_dataset = RViHSDDataset(train_df['text'].values, train_h_labels, train_n_labels,
#                                tokenizer, 128, augmenter, is_train=True)
# val_dataset   = RViHSDDataset(val_df['text'].values, val_h_labels, val_n_labels,
#                                tokenizer, 128, augmenter, is_train=False)
# train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_dataloader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
#
# SEEDS = [2026, 2027, 2028]
# ensemble_paths = []
# for seed in SEEDS:
#     path, f1 = train_one_model(seed, train_dataloader, val_dataloader, epochs=10)
#     ensemble_paths.append(path)
print("[*] Logic huấn luyện đã sẵn sàng (không tự động chạy trong notebook demo này).")